In [1]:
import xml.sax
from collections import defaultdict

KEEP_LANG = {"en", "da"}

concepts = defaultdict(lambda: {
    "en": None,
    "da": None,
    "broader": [],
    "narrower": []
})

current_concept = None
current_lang = None
collect_text = False
buffer = []

class ExtractHandler(xml.sax.ContentHandler):
    def startElement(self, name, attrs):
        global current_concept, current_lang, collect_text

        # Track concept
        if name == "skos:Concept":
            current_concept = attrs.get("rdf:about")

        # Track broader/narrower relations
        if name == "skos:broader":
            uri = attrs.get("rdf:resource")
            if current_concept and uri:
                concepts[current_concept]["broader"].append(uri)

        if name == "skos:narrower":
            uri = attrs.get("rdf:resource")
            if current_concept and uri:
                concepts[current_concept]["narrower"].append(uri)

        # Detect label-bearing element (SKOS or SKOS-XL literalForm)
        lang = attrs.get("xml:lang")
        if lang in KEEP_LANG:
            current_lang = lang
            collect_text = True
            buffer.clear()

    def characters(self, content):
        if collect_text:
            buffer.append(content)

    def endElement(self, name):
        global current_concept, current_lang, collect_text

        # Store collected label
        if collect_text and name.endswith("prefLabel") or name.endswith("literalForm"):
            text = "".join(buffer).strip()
            if current_concept and current_lang and text:
                concepts[current_concept][current_lang] = text

            collect_text = False
            current_lang = None
            buffer.clear()

        # End of concept
        if name == "skos:Concept":
            current_concept = None

parser = xml.sax.make_parser()
parser.setContentHandler(ExtractHandler())
parser.parse("esco-v1.2.0.rdf")

print("Concepts extracted:", len(concepts))

Concepts extracted: 17508


In [ ]:
import pyoxigraph
from rdflib import Graph, Namespace, Literal, URIRef

def fast_extract_esco():
    input_file = "esco-v1.2.0.rdf"
    output_file = "esco_framework_en_da.ttl"
    
    # Namespaces
    ESCO = "http://data.europa.eu/esco/model#"
    SKOS = "http://www.w3.org/2004/02/skos/core#"
    RDF = "http://www.w3.org/1999/02/22-rdf-syntax-ns#"
    DCT = "http://purl.org/dc/terms/"
    
    target_langs = {b"en", b"da"} # Oxigraph uses bytes for languages

    # 1. Initialize Oxigraph Store (on disk to save RAM)
    store = pyoxigraph.Store("temp_esco_store")
    
    print("Parsing RDF with Oxigraph...")
    with open(input_file, "rb") as f:
        store.load(f, "application/rdf+xml")

    # 2. Create an RDFLib graph for the output (it will be small)
    out_g = Graph()
    out_g.bind("esco", Namespace(ESCO))
    out_g.bind("skos", Namespace(SKOS))
    
    print("Extracting structural framework...")
    
    # Query for the 'Backbone' (Edges)
    # We use a SPARQL query inside Oxigraph to grab exactly what we need
    query = f"""
    PREFIX skos: <{SKOS}>
    PREFIX esco: <{ESCO}>
    PREFIX rdf: <{RDF}>
    SELECT ?s ?p ?o WHERE {{
        ?s ?p ?o .
        VALUES ?p {{ 
            skos:broader skos:narrower skos:related 
            esco:relatedEssentialSkill esco:relatedOptionalSkill 
            rdf:type 
        }}
    }}
    """
    
    for solution in store.query(query):
        out_g.add((URIRef(solution["s"].value), 
                   URIRef(solution["p"].value), 
                   URIRef(solution["o"].value)))

    # Query for Labels (Language Filtered)
    print("Extracting English and Danish labels...")
    label_query = f"""
    PREFIX skos: <{SKOS}>
    PREFIX dct: <{DCT}>
    SELECT ?s ?p ?o WHERE {{
        ?s ?p ?o .
        VALUES ?p {{ skos:prefLabel skos:altLabel dct:description }}
        FILTER (LANG(?o) = "en" || LANG(?o) = "da")
    }}
    """
    
    for solution in store.query(label_query):
        out_g.add((URIRef(solution["s"].value), 
                   URIRef(solution["p"].value), 
                   Literal(solution["o"].value, lang=solution["o"].language.decode('utf-8'))))

    print(f"Saving to {output_file}...")
    out_g.serialize(destination=output_file, format="turtle")
    print("Done! You can delete the 'temp_esco_store' folder now.")

if __name__ == "__main__":
    fast_extract_esco()

Parsing RDF with Oxigraph...


In [ ]:
from rdflib import Graph, Namespace
from rdflib.namespace import SKOS, RDF

def summarize_framework(file_path):
    g = Graph()
    print(f"Loading {file_path}...")
    g.parse(file_path, format="turtle")
    
    ESCO = Namespace("http://data.europa.eu/esco/model#")
    
    # 1. Total Triple Count
    print(f"\n--- General Overview ---")
    print(f"Total Triples: {len(g)}")

    # 2. Count by Entity Type
    query_types = """
    SELECT ?type (COUNT(?s) AS ?count)
    WHERE {
        ?s a ?type .
        FILTER (STRSTARTS(STR(?type), "http://data.europa.eu/esco/model#"))
    }
    GROUP BY ?type
    """
    print("\n--- Entity Distribution ---")
    for row in g.query(query_types):
        type_name = row.type.split('#')[-1]
        print(f"{type_name}: {row.count}")

    # 3. Check for Essential/Optional Skill links
    query_links = """
    SELECT (COUNT(?occ) AS ?linkCount)
    WHERE {
        { ?occ <http://data.europa.eu/esco/model#relatedEssentialSkill> ?skill }
        UNION
        { ?occ <http://data.europa.eu/esco/model#relatedOptionalSkill> ?skill }
    }
    """
    links = list(g.query(query_links))[0][0]
    print(f"\n--- Inference Links ---")
    print(f"Occupation-to-Skill Relationships: {links}")

    # 4. Language Verification (Sample)
    print("\n--- Language Check (Top 5 English/Danish Labels) ---")
    query_langs = """
    SELECT ?label
    WHERE {
        ?s skos:prefLabel ?label .
        FILTER (lang(?label) = 'da')
    }
    LIMIT 5
    """
    for row in g.query(query_langs):
        print(f"Danish Label: {row.label}")

if __name__ == "__main__":
    summarize_framework("esco_framework_en_da.ttl")

In [3]:
from rdflib import Graph, URIRef, Literal
from rdflib.namespace import SKOS, RDF
from tqdm.notebook import tqdm

g = Graph()

for uri, data in tqdm(concepts.items()):
    s = URIRef(uri)

    # Type
    g.add((s, RDF.type, SKOS.Concept))

    # Add labels for this concept
    if data["en"]:
        g.add((s, SKOS.prefLabel, Literal(data["en"], lang="en")))
    if data["da"]:
        g.add((s, SKOS.prefLabel, Literal(data["da"], lang="da")))

    # Hierarchy with embedded labels
    for b in data["broader"]:
        b_uri = URIRef(b)
        g.add((s, SKOS.broader, b_uri))

        # Add labels for the broader node itself
        b_data = concepts.get(b, {})
        if b_data.get("en"):
            g.add((b_uri, SKOS.prefLabel, Literal(b_data["en"], lang="en")))
        if b_data.get("da"):
            g.add((b_uri, SKOS.prefLabel, Literal(b_data["da"], lang="da")))

    # (Optional: also embed narrower, but broader is enough)
    
g.serialize("esco_cleaned.rdf", format="pretty-xml")

  0%|          | 0/17508 [00:00<?, ?it/s]

<Graph identifier=N8e4eb6ef83e5426896265ce00df036c3 (<class 'rdflib.graph.Graph'>)>

In [4]:
skill_prefix = "http://data.europa.eu/esco/skill/"

all_skills = {
    str(s)
    for s in g.subjects()
    if str(s).startswith(skill_prefix)
}

In [5]:
edges = {}

for s in all_skills:
    broader_parents = [
        str(o)
        for o in g.objects(s, SKOS.broader)
        if str(o).startswith(skill_prefix)
    ]
    edges[s] = broader_parents

In [6]:
root_skills = [s for s, parents in edges.items() if len(parents) == 0]

In [7]:
from rdflib.namespace import SKOS
from rdflib import Literal, URIRef

def get_en_label(uri):
    for lbl in g.objects(URIRef(uri), SKOS.prefLabel):
        if isinstance(lbl, Literal) and lbl.language == 'en':
            return str(lbl)
    return None

# roots from your edges dict
root_skills = [s for s, parents in edges.items() if len(parents) == 0]

root_names = [get_en_label(s) for s in root_skills]
root_names = [name for name in root_names if name]  # drop None

len(root_names)

13746

In [11]:
edges

{'http://data.europa.eu/esco/skill/478b826b-21a3-464d-a697-faef4faa7dc1': [],
 'http://data.europa.eu/esco/skill/135488ce-f5ba-4238-b7d1-8a1197664ab6': [],
 'http://data.europa.eu/esco/skill/f14bd327-7ddc-4e83-a06b-cb0214c76183': [],
 'http://data.europa.eu/esco/skill/c9adb847-a10b-4eeb-b4d9-15e1b4e732a0': [],
 'http://data.europa.eu/esco/skill/4377dd26-a0d6-497c-9ffe-f59308d36648': [],
 'http://data.europa.eu/esco/skill/df794a74-0392-4178-918b-e38e68487325': [],
 'http://data.europa.eu/esco/skill/1bd1b8e7-246d-4481-9f26-d62abb7edfb6': [],
 'http://data.europa.eu/esco/skill/11cbcf7f-8dc3-4c9a-9ec1-37f51b5cbc19': [],
 'http://data.europa.eu/esco/skill/8f1efe4e-5f92-43b0-a329-3c17c1460a37': [],
 'http://data.europa.eu/esco/skill/609a8ac1-9d29-4237-9886-596dbbe7ca8a': [],
 'http://data.europa.eu/esco/skill/16da7f95-bd00-4a5c-b837-bf1c2a5a9f15': [],
 'http://data.europa.eu/esco/skill/c9b97523-d520-4494-aa60-e3bc9ef0d8a4': [],
 'http://data.europa.eu/esco/skill/05338a70-1ea1-4c33-a886-1c4f0

In [10]:
def print_tree(G, node, prefix=""):
    label = G.nodes[node]["en"] or "(no label)"
    
    # Print this node
    print(prefix + label)

    # Collect children (narrower concepts)
    children = list(G.successors(node))
    children.sort(key=lambda c: G.nodes[c]["en"] or "")

    for i, child in enumerate(children):
        is_last = (i == len(children) - 1)
        branch = "└── " if is_last else "├── "
        extension = "    " if is_last else "│   "
        print_tree(child, prefix + branch)

# Print only one top-level tree first
print_tree(g, root_names[0])

AttributeError: 'Graph' object has no attribute 'nodes'